## tl;dr

The May 22–August 10, 2026 Search Console export contains 8 impressions and 1 click. The sample is too small for trend or CTR optimization. The only visible query is **shentel speed test**, so the first content action is a focused Shentel results and troubleshooting guide.

## Context & Methods

This diagnostic notebook validates the totals and dimensions in a standard Google Search Console ZIP export. Set `GSC_ZIP_PATH` to the export before executing.

### Key Assumptions

- The ZIP is an unmodified Search Console Performance export.
- Dates use Search Console's reporting convention.
- Query rows can be hidden by Google's privacy filtering, so query-table sums are not expected to equal chart totals.
- Eight impressions are insufficient for reliable trend, device, or CTR conclusions.

## Data

In [ ]:
from io import BytesIO
import os
from zipfile import ZipFile

import pandas as pd

zip_path = os.environ['GSC_ZIP_PATH']
with ZipFile(zip_path) as archive:
    tables = {
        name.rsplit('/', 1)[-1].replace('.csv', ''): pd.read_csv(BytesIO(archive.read(name)))
        for name in archive.namelist()
        if name.endswith('.csv')
    }

sorted(tables)

In [ ]:
required = {'Chart', 'Queries', 'Pages', 'Countries', 'Devices', 'Filters'}
missing = required.difference(tables)
assert not missing, f'Missing expected exports: {sorted(missing)}'

chart = tables['Chart'].copy()
chart['Date'] = pd.to_datetime(chart['Date'])
assert chart['Date'].is_monotonic_increasing
assert (chart[['Clicks', 'Impressions']] >= 0).all().all()

summary = pd.Series({
    'start_date': chart['Date'].min().date().isoformat(),
    'end_date': chart['Date'].max().date().isoformat(),
    'days_in_export': len(chart),
    'days_with_impressions': int((chart['Impressions'] > 0).sum()),
    'clicks': int(chart['Clicks'].sum()),
    'impressions': int(chart['Impressions'].sum()),
    'ctr': chart['Clicks'].sum() / chart['Impressions'].sum(),
    'visible_query_rows': len(tables['Queries']),
})
summary

## Results

In [ ]:
active_days = chart.loc[chart['Impressions'] > 0, ['Date', 'Clicks', 'Impressions', 'CTR', 'Position']]
active_days

In [ ]:
tables['Queries']

In [ ]:
tables['Pages']

## Takeaways

- Treat this export as early discovery evidence, not a ranking baseline.
- Build the first new guide around `shentel speed test`, the only visible demand signal.
- Do not infer a desktop preference, CTR problem, or traffic trend from eight impressions.
- Re-run the same notebook on a 28-day export after deployment and compare only once impressions are sustained across multiple weeks and landing pages.